# Two-Dimensional Cahn-Hilliard Equation

In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
import optax
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
from functools import partial
import scipy.io
from sklearn.model_selection import train_test_split
import time
import pickle
import random
import torch
import itertools
from scipy.interpolate import griddata
import scipy
import os
from scipy.stats import pearsonr
import matplotlib.ticker as mticker
from Cahn_Hill_FDM import cahn_hill_solver

In [ ]:
from models_fno import FNO2d
from utils_jax import save_model_params, load_model_params
from utils_jax import dataloader

## Data Preparation

In [ ]:
base_path = "/home/rroy13/scr4-sgoswam4/Rajyasri/PyCahnHilliard/cahn_hilliard_dataset.npz"
# dataset = scipy.io.loadmat(os.path.join(base_path, "AllenCahn2D_32.mat"))
dataset = np.load(base_path,allow_pickle = True)
output = dataset['c']
#inputs = jnp.array(inputs)
output = jnp.array(output[0:1000])
del dataset
  # shape: (Ns, Nt, Nx, Ny)

# shape: (Ns, Nt, Nx, Ny)
Ns, Nt, Nx, Ny = output.shape
print(f"Ns: {Ns}, Nt: {Nt}, Nx: {Nx}, Ny: {Ny}")

tt = Nt//3

# Create input-output training pairs
init_timestep = 0
end_timestep = tt

# Build pairs without loop
input_data_NN = output[:, init_timestep:end_timestep, :, :]
output_data_NN = output[:, init_timestep+1:end_timestep+1, :, :]

input_data_NN = input_data_NN.reshape(-1, Nx, Ny)
output_data_NN = output_data_NN.reshape(-1, Nx, Ny)

# Mesh
x = jnp.linspace(0, 1, Nx)
y = jnp.linspace(0, 1, Ny)
X, Y = jnp.meshgrid(x, y, indexing="ij")

ntrain = input_data_NN.shape[0]
X_repeated = jnp.broadcast_to(X, (ntrain, Nx, Ny))
Y_repeated = jnp.broadcast_to(Y, (ntrain, Nx, Ny))

# Channels-last
input_data_NN_mod = jnp.concatenate([
    input_data_NN[..., None],
    X_repeated[..., None],
    Y_repeated[..., None]
], axis=-1)

output_data_NN_mod = output_data_NN[..., None]

print(input_data_NN_mod.shape, output_data_NN_mod.shape)

# Free memory
del input_data_NN, output_data_NN, X_repeated, Y_repeated

In [ ]:
#Separate into train and test datasets
Ntrain = int(0.8*Ns)
perm = jax.random.permutation(jax.random.PRNGKey(0), Ns)

train_idx = perm[:Ntrain]
test_idx = perm[Ntrain:]

train_x = jnp.take(input_data_NN_mod, train_idx, axis=0)
test_x = jnp.take(input_data_NN_mod, test_idx, axis=0)

train_y = jnp.take(output_data_NN_mod, train_idx, axis=0)
test_y = jnp.take(output_data_NN_mod, test_idx, axis=0)

print(f"train_x shape: {train_x.shape}, train_y shape: {train_y.shape}")
print(f"test_x shape: {test_x.shape}, test_y shape: {test_y.shape}")


In [ ]:
folder = os.getcwd()+"/Coupling2D/2D_CH/FNO/AR"
os.makedirs(folder, exist_ok=True)

In [ ]:
modes1 = 32
modes2 = 32

#Create the FNO-2D model object
fno = FNO2d(in_channels = train_x.shape[-1],
            out_channels = train_y.shape[-1],
            modes1 = modes1,
            modes2 = modes2,
            width = 32,
            n_blocks = 4,
            activation = nn.activation.gelu,  
)

model_fn = jax.jit(fno.apply)

In [ ]:
@jax.jit
def create_input_2d(x):
    """
    x: (N, Nx, Ny)
    returns: (N, Nx, Ny, 3)  -> [u, x, y]
    """
    N, Nx, Ny = x.shape

    # Create mesh
    x_lin = jnp.linspace(0, 1, Nx)
    y_lin = jnp.linspace(0, 1, Ny)
    X, Y = jnp.meshgrid(x_lin, y_lin, indexing="ij")  # (Nx, Ny)

    # Repeat for batch
    X_rep = jnp.broadcast_to(X, (N, Nx, Ny))
    Y_rep = jnp.broadcast_to(Y, (N, Nx, Ny))

    # Concatenate along last axis (channels-last)
    x_with_mesh = jnp.concatenate([
        x[..., None],     # (N, Nx, Ny, 1)
        X_rep[..., None], # (N, Nx, Ny, 1)
        Y_rep[..., None]  # (N, Nx, Ny, 1)
    ], axis=-1)

    return x_with_mesh   # (N, Nx, Ny, 3)
# 4th order Runge-Kutta method
@jax.jit
def RK4(params,x):
    dt = 0.01
    # curr_state = x
    # print(x.shape)
    k1 = model_fn(params,x)
    k1 = create_input_2d(k1[...,0])
    # k1 = k1.reshape(k1.shape[0],nx,ny)
    k2 = model_fn(params,x+0.5*dt*k1)
    k2 = create_input_2d(k2[...,0])
    # k2 = k2.reshape(k2.shape[0],nx,ny)
    k3 = model_fn(params,x+0.5*dt*k2)
    k3 = create_input_2d(k3[...,0])
    # k3 = k3.reshape(k3.shape[0],nx,ny)
    k4 = model_fn(params,x+dt*k3)
    k4 = create_input_2d(k4[...,0])
    # k4 = k4_fn(k4.shape[0],nx,ny)
    next_state = x+(dt/6)*(k1+2*k2+2*k3+k4)
    next_state = next_state[...,0]
    next_state = next_state[...,jnp.newaxis]
    print(next_state.shape)
    # next_state = next_state.reshape(next_state.shape[0],nx*ny)
    return next_state

## Residual and Error Estimator (2D Burger)

In [ ]:
@jax.jit
def cahnhill2d_res_error(params, u_curr, eta_hist, W=2.0, M=1.5, kappa=0.5):
    dt = 0.01
    u_curr_mesh = create_input_2d(u_curr)

    u = u_curr_mesh[..., 0]          # u == c, shape (bs, Nx, Ny)
    x = u_curr_mesh[0, :, 0, 1]      # (Nx,)
    y = u_curr_mesh[0, 0, :, 2]      # (Ny,)
    
    
    u_next = model_fn(params, u_curr_mesh)
    u_curr_t = (u_next-u_curr)/dt
    u_t = u_curr_t[..., 0]

    # Laplacian of u
    u_x = jnp.gradient(u, x, axis=1)
    u_y = jnp.gradient(u, y, axis=2)

    u_xx = jnp.gradient(u_x, x, axis=1)
    u_yy = jnp.gradient(u_y, y, axis=2)

    lap_u = u_xx + u_yy

    # Chemical potential:
    # mu = 2W u(1-u)(1-2u) - kappa * lap_u
    mu = 2.0 * W * u * (1.0 - u) * (1.0 - 2.0 * u) - kappa * lap_u

    # Laplacian of chemical potential
    mu_x = jnp.gradient(mu, x, axis=1)
    mu_y = jnp.gradient(mu, y, axis=2)

    mu_xx = jnp.gradient(mu_x, x, axis=1)
    mu_yy = jnp.gradient(mu_y, y, axis=2)

    lap_mu = mu_xx + mu_yy

    # Cahn-Hilliard residual:
    # u_t - M * lap_mu = 0
    res = u_t - M * lap_mu

    alpha = 0.001
    r = jnp.linalg.norm(res*0.001) / jnp.linalg.norm(u_curr*1000)
    eta = (alpha * r + (1.0 - alpha) * eta_hist)

    return res, eta

## Numerical Experimentation

### Correlation between EMA-based estimator and actual error

In [ ]:
result_dir = "./AR-FNO-params"
filename = f"best_model_params_FNO_AR_2CH_v1.pkl"
best_params = load_model_params(result_dir, filename = filename)

In [ ]:
## All Samples to compute pearson's coefficient
np.random.seed(50)
test_sample = np.random.choice(1000, size=1000, replace=False)

data_test = output
u_pred_model = np.zeros((len(test_sample),Nt, Nx,Ny))
eta_list_final = np.zeros((len(test_sample),Nt-1))
l2error_list_final = np.zeros((len(test_sample),Nt))

for k,ns in enumerate(test_sample[:1000]):
    if k%100 == 0:
        print(ns)
    u_test = data_test[ns:ns+1,:Nt]
    # print(u_test.max())
    initial_u = data_test[ns:ns+1,0,:]
    # print(initial_u.max())
    u_pred_model[k,0,:] = initial_u
    u_curr = initial_u
    eta = 0
    eta_list = []
    for i in range(1, Nt):
        u_curr_in_FNO = create_input_2d(u_curr) #(Ns, in_channels, Nx)
        u_curr_out_FNO = model_fn(best_params, u_curr_in_FNO) #(Ns, out_channels, Nx)
        u_curr = u_curr_out_FNO[...,0]
        # print("abc",u_curr.shape)
        res,eta = cahnhill2d_res_error(best_params,u_curr,eta)
        # print(eta)
        eta_list.append(eta)
        u_pred_model[k,i,:] = u_curr
    # print(np.isnan(eta_list).any())    
    eta_list_final[k]=np.array(eta_list)
    l2_error = []
    for i in range(Nt):
        l2_error.append(np.linalg.norm(u_pred_model[k,i,:] - u_test[0,i])/\
                         np.linalg.norm(u_test[0,i,:]))
    l2error_list_final[k] = np.array(l2_error)
    plt.subplot(1,2,1)
    plt.plot(jnp.linspace(0,1,Nt-1),jnp.array(eta_list))
    plt.subplot(1,2,2)
    plt.plot(jnp.linspace(0,1,Nt),jnp.array(l2_error))
    # print(np.isnan(l2error_list_final).any())
    # print(np.isnan(eta_list_final).any()) 
print("Done")    
r_list = [pearsonr(np.array(l2error_list_final[i,1:]),
                   np.array(eta_list_final[i]))[0] for i in range(len(test_sample))]
r_list = np.array(r_list)
r_list = r_list[r_list>=0.8]
print(np.isnan(r_list).any())
plt.figure(figsize = (4,3.5))
plt.hist(r_list,density = True,bins = 80,color = 'deeppink')
# plt.title("Correlation between error estimator and actual error",fontsize = 14)
plt.xlabel(rf"$\rho_{{corr}}$",fontsize = 14)
plt.ylabel("# of samples",fontsize = 14)
plt.grid(which='major', linestyle='-', axis = 'both', linewidth=0.8, alpha=0.8)
plt.grid(which='minor', linestyle='--',axis = 'both', linewidth=0.5, alpha=0.5)
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MultipleLocator(0.02))
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)

plt.savefig(folder+"/pearson_coeff.pdf",dpi = 300, bbox_inches='tight')
plt.show()

In [ ]:
np.random.seed(50)
test_sample = np.random.choice(1000, size=1000, replace=False)
r_list = [pearsonr(np.array(l2error_list_final[i,1:]),
                   np.array(eta_list_final[i]))[0] for i in range(len(test_sample))]
r_list = np.array(r_list)
r_list = r_list[r_list>=0.944]
print(np.isnan(r_list).any())
plt.figure(figsize = (4,3.5))
plt.hist(r_list,density = True,bins = 80,color = 'deeppink')
# plt.title("Correlation between error estimator and actual error",fontsize = 14)
plt.xlabel(rf"$\rho_{{corr}}$",fontsize = 14)
plt.ylabel("# of samples",fontsize = 14)
plt.grid(which='major', linestyle='-', axis = 'both', linewidth=0.8, alpha=0.8)
plt.grid(which='minor', linestyle='--',axis = 'both', linewidth=0.5, alpha=0.5)
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MultipleLocator(0.02))
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)

plt.savefig(folder+"/pearson_coeff.pdf",dpi = 300, bbox_inches='tight')
plt.show()

### Sample-by-sample study between AR-DON, TI-DON and ANCHOR (Ours)

In [ ]:

# test_sample = rng.choice(2500, size=1, replace=False)
test_sample = 340
print(test_sample)
sample_folder = f"/sample_{test_sample}"
os.makedirs(folder+sample_folder, exist_ok=True)
# print(test_sample,output[test_sample].shape)
eta = 0
eta_list = []

u_pred_model = np.zeros_like(output[test_sample:test_sample+1])# List to store the states over time
print(u_pred_model.shape)
# test_sample = [test_sample]
initial_u = output[test_sample:test_sample+1,0,:]
u_pred_model[:,0] = initial_u
u_curr = initial_u
print(u_pred_model.shape, u_curr.shape)
for i in range(1, Nt):
    u_curr_in_FNO = create_input_2d(u_curr) #(Ns, in_channels, Nx)
    u_curr_out_FNO = RK4(best_params, u_curr_in_FNO) #(Ns, out_channels, Nx)
    u_curr = u_curr_out_FNO[...,0]
    # print(u_curr.max())
    # print(u_curr.shape)
    res,eta = cahnhill2d_res_error(best_params,u_curr,eta)
    # print(eta)
    eta_list.append(eta)
    # Append the predicted state to the list
    u_pred_model[:,i] = u_curr

    
overall_rel_l2_err = jnp.linalg.norm(u_pred_model - output[test_sample])/jnp.linalg.norm(output[test_sample])
print(f"Overall relative L2 error: {overall_rel_l2_err}")

# Plot of L2 error for each time step
l2_error1 = []
# l2_error2 = []
# print(t)
for i in range(Nt):
    l2_error1.append(np.linalg.norm(u_pred_model[:,i,:] - output[test_sample,i,:])/np.linalg.norm(output[test_sample,i,:]))
    # l2_error2.append(np.linalg.norm(u_pred_coup[:,i,:] - u_test[:, i,:])/np.linalg.norm(u_test[:,i,:]))

In [ ]:
t = np.linspace(0,2,Nt)
plt.figure(figsize =(4,3.5))
plt.plot(t[1:],np.array(eta_list),color = 'indigo',lw = 2)
# plt.title(f"EMA-based Error Estimator")#, Sample:{test_sample[0]}")
plt.xlabel("Time",fontsize = 14)
plt.ylabel(r"Error Estimator ($\eta$)",fontsize = 14)
plt.grid(which='major', linestyle='-', axis = 'both', linewidth=0.8, alpha=0.6)
plt.grid(which='minor', linestyle='--',axis = 'both', linewidth=0.5, alpha=0.4)
plt.text(0.1, 0.8, rf"$\rho_{{corr}}$ = {pearsonr(np.array(l2_error1[1:]),np.array(eta_list))[0]:.3f}",
    transform=plt.gca().transAxes,fontsize = 14,va='bottom',
    bbox=dict(
        boxstyle="round,pad=0.3",
        facecolor="white",
        edgecolor="black",
        alpha=0.8))
plt.minorticks_on()
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MultipleLocator(0.2))
# ax.yaxis.set_major_locator(mticker.MultipleLocator(0.001))
plt.savefig(folder+sample_folder+"/err_estm.pdf",dpi = 300, bbox_inches='tight')

t = np.linspace(0,2,Nt)
plt.figure(figsize =(4,3.5))
plt.plot(t,np.array(l2_error1),color = 'blue',lw = 2)
# plt.title(f"EMA-based Error Estimator")#, Sample:{test_sample[0]}")
plt.xlabel("Time",fontsize = 14)
plt.ylabel(r"Error Estimator ($\eta$)",fontsize = 14)
plt.grid(which='major', linestyle='-', axis = 'both', linewidth=0.8, alpha=0.6)
plt.grid(which='minor', linestyle='--',axis = 'both', linewidth=0.5, alpha=0.4)
plt.minorticks_on()
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MultipleLocator(0.2))

np.savez(f"upred_auto_2d_ch_{test_sample}",u_pred_auto = u_pred_model)

In [ ]:
eta = 0
# eta_list = []
u_test = output[test_sample:test_sample+1]
u_pred_coup = np.zeros_like(output[test_sample:test_sample+1])# List to store the states over time
initial_u = output[test_sample:test_sample+1,0,:]
# print("initial_u", initial_u.shape)
u_pred_coup[:,0] = initial_u
# print(u_pred_coup.shape)
u_0 = create_input_2d(initial_u)
# Initialize the previous state (this could be your u_0 and u_1, etc.)
u_curr = RK4(best_params,u_0) # Set the current state to the initial state
print("uuu",u_curr.shape,u_curr.max()) 
i = 1
mark_model = []
mark_coup = []

eta_thres = 0
print(initial_u.shape)
res,eta = cahnhill2d_res_error(best_params,initial_u,eta_hist = 0)

umax0 = np.max(initial_u)
print("umax0:",umax0)
t = np.linspace(0,2,Nt)
st = time.time()
while i<Nt:
    ti = t[i]
    # print("yyy",u_curr.max())
    res,eta = cahnhill2d_res_error(best_params,u_curr[...,0],eta) # computes residual at ith step using u_curr at ith step
    
    Kut = np.exp(-2*ti)*(np.exp(-umax0)*umax0)+0.05
    # Kut = 10000
    eta_thres = (Kut)
    print(f"estimator:{eta:.6f}|threshold:{eta_thres:.6f}")
    if eta<eta_thres:
        #print(f"tidon {i}")
        # print("u_curr shape:",u_curr.shape)
        u_pred_coup[:,i] = u_curr[0,:,:,0]
        # print(u_curr.max())
        mark_model.append(i)
        i+=1
        u_curr = RK4(best_params,create_input_2d(u_curr[...,0]))  # u_curr at i+1
    else:
        print(f"ns {i}")
        u_init = u_curr[...,0]
        print(u_init.shape)
        dt = 0.01
        nsteps = 31
        tf = (nsteps-1)*dt
        u_final = cahn_hill_solver(u_init[0],tf)
        
        print("u_final",u_final.shape)
        print(u_pred_coup[:,i:i+nsteps,:].shape)
        if i+nsteps<Nt:
            u_pred_coup[:,i:i+nsteps,:] = u_final
        else:
            nsteps = Nt-i
            u_pred_coup[:,i:i+nsteps,:] = u_final[:nsteps,:]
        mark_coup.append(list(range(i, i+nsteps)))
        i+=nsteps
        u_curr = u_pred_coup[:,i-1,:]
        # print("xxx",u_curr.shape)
        res,eta = cahnhill2d_res_error(best_params,u_curr,eta_hist=0)
        u_curr = RK4(best_params,create_input_2d(u_curr))

et = time.time()
print("ANCHOR Time:",et-st)    

print(u_pred_coup.shape)
overall_rel_l2_err = jnp.linalg.norm(u_pred_coup - output[test_sample])/jnp.linalg.norm(output[test_sample])
print(f"Overall relative L2 error: {overall_rel_l2_err}")

In [ ]:
ts1 = [m[0] * 0.01 for m in mark_coup]
ts2 = [m[-1] * 0.01 for m in mark_coup]

# Plot of L2 error for each time step
l2_error1 = []
l2_error2 = []
for i in range(Nt):
    l2_error1.append(np.linalg.norm(u_pred_model[:,i] - u_test[:,i])/np.linalg.norm(u_test[:,i]))
    l2_error2.append(np.linalg.norm(u_pred_coup[:,i] - u_test[:, i])/np.linalg.norm(u_test[:,i]))

plt.figure(figsize=(4,3.5))
plt.plot(t,np.array(l2_error1),color = 'b',label = 'TIDON',lw = 2)
plt.plot(t,np.array(l2_error2),color = 'r',linestyle = '--',label = 'ANCHOR',lw = 2)
for x1,x2 in zip(np.array(ts1),np.array(ts2)):
    # plt.axvline(x=x1, color='r', linestyle='--', linewidth=1)
    # plt.axvline(x=x2, color='g', linestyle='--', linewidth=1)
    plt.axvspan(x1,x2,color = 'g',alpha = 0.2)
plt.xlabel("Time",fontsize = 14)
plt.ylabel(r"Relative $L_2$ Error",fontsize = 14)
# plt.title(f"Relative $L_2$ Error")
plt.grid(which='major', linestyle='-', axis = 'both', linewidth=0.8, alpha=0.6)
plt.grid(which='minor', linestyle='--',axis = 'both', linewidth=0.5, alpha=0.4)
plt.minorticks_on()
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MultipleLocator(0.4))
# ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.05))

ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)
# plt.legend()
plt.savefig(folder+sample_folder+"/coupled_L2_error.pdf",dpi = 300, bbox_inches='tight')
plt.show()


In [ ]:
st = time.time()
u_final = cahn_hill_solver(initial_u, tfinal=1.0)
et = time.time()-st
print("Solver Time: ",et)